 Rusty Bargain used car sales service is developing an app to attract new customers. In that app, you can quickly find out the market value of your car. You have access to historical data: technical specifications, trim versions, and prices. You need to build the model to determine the value.



 Rusty Bargain is interested in:



 - the quality of the prediction;

 - the speed of the prediction;

 - the time required for training

In [1]:
# %%
import matplotlib.pyplot as plt
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np


 # 1. Data preparation

 ### Downloading

In [2]:
# %%
data = pd.read_csv('https://code.s3.yandex.net/datasets/autos.csv')
print(data.shape)
data.head()


(354369, 16)


,DateCrawled,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Kilometer,RegistrationMonth,FuelType,Brand,Repaired,DateCreated,NumberOfPictures,PostalCode,LastSeen
0,2016-03-24 11:52:17,480,NaN,1993,manual,0,golf,150000,0,petrol,volkswagen,NaN,2016-03-24 00:00:00,0,70435,2016-04-07 03:16:57
1,2016-03-24 10:58:45,18300,coupe,2011,manual,190,NaN,125000,5,gasoline,audi,yes,2016-03-24 00:00:00,0,66954,2016-04-07 01:46:50
2,2016-03-14 12:52:21,9800,suv,2004,auto,163,grand,125000,8,gasoline,jeep,NaN,2016-03-14 00:00:00,0,90480,2016-04-05 12:47:46
3,2016-03-17 16:54:04,1500,small,2001,manual,75,golf,150000,6,petrol,volkswagen,no,2016-03-17 00:00:00,0,91074,2016-03-17 17:40:17
4,2016-03-31 17:25:20,3600,small,2008,manual,69,fabia,90000,7,gasoline,skoda,no,2016-03-31 00:00:00,0,60437,2016-04-06 10:17:21


 ### Preprocessing

 Unnecessary features (can't be used in the product):



 - dates

 - zip-code

In [3]:
# %%
data = data.drop(['DateCrawled', 'DateCreated', 'PostalCode', 'LastSeen'], axis=1)
data.head()


,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Kilometer,RegistrationMonth,FuelType,Brand,Repaired,NumberOfPictures
0,480,NaN,1993,manual,0,golf,150000,0,petrol,volkswagen,NaN,0
1,18300,coupe,2011,manual,190,NaN,125000,5,gasoline,audi,yes,0
2,9800,suv,2004,auto,163,grand,125000,8,gasoline,jeep,NaN,0
3,1500,small,2001,manual,75,golf,150000,6,petrol,volkswagen,no,0
4,3600,small,2008,manual,69,fabia,90000,7,gasoline,skoda,no,0


In [4]:
# %%
data['NumberOfPictures'].value_counts()


NumberOfPictures
0    354369
Name: count, dtype: int64

 Deleting constant feature

In [5]:
# %%
data = data.drop(['NumberOfPictures'], axis=1)
data.head()


,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Kilometer,RegistrationMonth,FuelType,Brand,Repaired
0,480,NaN,1993,manual,0,golf,150000,0,petrol,volkswagen,NaN
1,18300,coupe,2011,manual,190,NaN,125000,5,gasoline,audi,yes
2,9800,suv,2004,auto,163,grand,125000,8,gasoline,jeep,NaN
3,1500,small,2001,manual,75,golf,150000,6,petrol,volkswagen,no
4,3600,small,2008,manual,69,fabia,90000,7,gasoline,skoda,no


In [6]:
# %%
data.describe()


,Price,RegistrationYear,Power,Kilometer,RegistrationMonth
count,354369.000000,354369.000000,354369.000000,354369.000000,354369.000000
mean,4416.656776,2004.234448,110.094337,128211.172535,5.714645
std,4514.158514,90.227958,189.850405,37905.341530,3.726421
min,0.000000,1000.000000,0.000000,5000.000000,0.000000
25%,1050.000000,1999.000000,69.000000,125000.000000,3.000000
50%,2700.000000,2003.000000,105.000000,150000.000000,6.000000
75%,6400.000000,2008.000000,143.000000,150000.000000,9.000000
max,20000.000000,9999.000000,20000.000000,150000.000000,12.000000


 The registration year of 1000 and 9999 is clearly incorrect, delete entries with incorrect values.



 Power cannot be equal to 0.

In [7]:
# %%
data = data[data['RegistrationYear'] < 2050]
data = data[data['RegistrationYear'] > 1900]
data = data[data['Power'] != 0]

data.reset_index()
data.shape


(314100, 11)

In [8]:
# %%
data.isna().sum(axis=0)


Price                    0
VehicleType          22818
RegistrationYear         0
Gearbox               6501
Power                    0
Model                13395
Kilometer                0
RegistrationMonth        0
FuelType             21212
Brand                    0
Repaired             49708
dtype: int64

 Missing values are only in categorical columns. They can be filled with a new value "unknown".

In [9]:
# %%
data = data.fillna('unknown')
data.isna().sum(axis=0)


Price                0
VehicleType          0
RegistrationYear     0
Gearbox              0
Power                0
Model                0
Kilometer            0
RegistrationMonth    0
FuelType             0
Brand                0
Repaired             0
dtype: int64

 ### Feature encoding

In [12]:
# %%
categorical_features = [
    'VehicleType',
    'Gearbox', 
    'Model',
    'FuelType', 
    'Brand',
    'Repaired', 
]


 **OHE-encoding**



 Here, some features will have to be removed due to the large number of values

In [14]:
# %%
for feature in categorical_features:
    print(data[feature].value_counts())


VehicleType
sedan          84852
small          71522
wagon          60508
bus            26530
unknown        22818
convertible    19034
coupe          15078
suv            11115
other           2643
Name: count, dtype: int64
Gearbox
manual     246097
auto        61502
unknown      6501
Name: count, dtype: int64
Model
golf                  26760
other                 21248
3er                   18226
unknown               13395
polo                  11455
                      ...  
i3                        5
samara                    5
rangerover                3
serie_3                   3
range_rover_evoque        2
Name: count, Length: 250, dtype: int64
FuelType
petrol      196358
gasoline     90723
unknown      21212
lpg           4913
cng            506
hybrid         207
other          103
electric        78
Name: count, dtype: int64
Brand
volkswagen        68805
opel              34902
bmw               33876
mercedes_benz     28512
audi              27075
ford              2

In [15]:
# %%
data_ohe = data.drop(['Model', 'Brand'], axis=1)
data_ohe = pd.get_dummies(data_ohe)
print(data_ohe.shape)
data_ohe.head()


(314100, 28)


,Price,RegistrationYear,Power,Kilometer,RegistrationMonth,VehicleType_bus,VehicleType_convertible,VehicleType_coupe,VehicleType_other,VehicleType_sedan,VehicleType_small,VehicleType_suv,VehicleType_unknown,VehicleType_wagon,Gearbox_auto,Gearbox_manual,Gearbox_unknown,FuelType_cng,FuelType_electric,FuelType_gasoline,FuelType_hybrid,FuelType_lpg,FuelType_other,FuelType_petrol,FuelType_unknown,Repaired_no,Repaired_unknown,Repaired_yes
1,18300,2011,190,125000,5,False,False,True,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,True
2,9800,2004,163,125000,8,False,False,False,False,False,False,True,False,False,True,False,False,False,False,True,False,False,False,False,False,False,True,False
3,1500,2001,75,150000,6,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,True,False,True,False,False
4,3600,2008,69,90000,7,False,False,False,False,False,True,False,False,False,False,True,False,False,False,True,False,False,False,False,False,True,False,False
5,650,1995,102,150000,10,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,True


 Ordinal encoding

In [16]:
# %%
from sklearn.preprocessing import OrdinalEncoder

data[categorical_features] = OrdinalEncoder().fit_transform(data[categorical_features])

data.head()


,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Kilometer,RegistrationMonth,FuelType,Brand,Repaired
1,18300,2.0,2011,1.0,190,227.0,125000,5,2.0,1.0,2.0
2,9800,6.0,2004,0.0,163,117.0,125000,8,2.0,14.0,1.0
3,1500,5.0,2001,1.0,75,116.0,150000,6,6.0,38.0,0.0
4,3600,5.0,2008,1.0,69,101.0,90000,7,2.0,31.0,0.0
5,650,4.0,1995,1.0,102,11.0,150000,10,6.0,2.0,2.0


 ### Split into train-validation-test

In [17]:
# %%
from sklearn.model_selection import train_test_split

index_train_valid, index_test = train_test_split(data.index, test_size=0.2, random_state=12345)
index_train, index_valid = train_test_split(index_train_valid, test_size=0.25, random_state=54321)

data_train = data.loc[index_train]
data_valid = data.loc[index_valid]
data_test = data.loc[index_test]

data_ohe_train = data_ohe.loc[index_train]
data_ohe_valid = data_ohe.loc[index_valid]
data_ohe_test = data_ohe.loc[index_test]

print(data_train.shape)
print(data_valid.shape)
print(data_test.shape)

print(data_ohe_train.shape)
print(data_ohe_valid.shape)
print(data_ohe_test.shape)


(188460, 11)
(62820, 11)
(62820, 11)
(188460, 28)
(62820, 28)
(62820, 28)


 # 2. Model training

In [18]:
# %%
from sklearn.metrics import mean_squared_error

def rmse(y, a):
    return mean_squared_error(y, a)**0.5


 Constant model

In [19]:
# %%
pred_mean = np.ones(data['Price'].shape) * data['Price'].mean()
print(rmse(data['Price'], pred_mean))


4590.240919951061


 ### Models with OHE

In [20]:
# %%
features_train = data_ohe_train.drop(['Price'], axis=1)
target_train = data_ohe_train['Price']
features_valid = data_ohe_valid.drop(['Price'], axis=1)
target_valid = data_ohe_valid['Price']
features_test = data_ohe_test.drop(['Price'], axis=1)
target_test = data_ohe_test['Price']


 **Linear regression**

In [21]:
# %%

from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(features_train, target_train)


LinearRegression()

In [22]:
# %%

pred_train = model.predict(features_train)
pred_valid = model.predict(features_valid)
pred_test = model.predict(features_test)


In [23]:
# %%
print("Train RMSE:", rmse(target_train, pred_train).round(5))
print("Valid RMSE:", rmse(target_valid, pred_valid).round(5))
print("Test RMSE: ", rmse(target_test, pred_test).round(5))


Train RMSE: 3238.30929
Valid RMSE: 3216.58303
Test RMSE:  3223.39036


 ### Models with ordinal encoding

In [24]:
# %%
features_train = data_train.drop(['Price'], axis=1)
target_train = data_train['Price']
features_valid = data_valid.drop(['Price'], axis=1)
target_valid = data_valid['Price']
features_test = data_test.drop(['Price'], axis=1)
target_test = data_test['Price']


 **Random forrest**

In [25]:
# %%
from sklearn.ensemble import RandomForestRegressor

for depth in [1, 2, 4, 6, 8, None]:
    model = RandomForestRegressor(max_depth=depth, n_estimators=100)
    model.fit(features_train, target_train)
    
    pred_train = model.predict(features_train)
    pred_valid = model.predict(features_valid)
    print("Depth:", depth)
    print("Train RMSE:", rmse(target_train, pred_train).round(5))
    print("Valid RMSE:", rmse(target_valid, pred_valid).round(5))


Depth: 1
Train RMSE: 3790.22037
Valid RMSE: 3782.29044
Depth: 2
Train RMSE: 3330.0806
Valid RMSE: 3319.26108
Depth: 4
Train RMSE: 2688.14011
Valid RMSE: 2674.87205
Depth: 6
Train RMSE: 2335.3801
Valid RMSE: 2336.725
Depth: 8
Train RMSE: 2099.39466
Valid RMSE: 2124.44798
Depth: None
Train RMSE: 771.17461
Valid RMSE: 1733.3114


In [26]:
# %%

from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, max_depth=None)
model.fit(features_train, target_train)


RandomForestRegressor()

In [27]:
# %%

pred_train = model.predict(features_train)
pred_valid = model.predict(features_valid)
pred_test = model.predict(features_test)


In [28]:
# %%
print("Train RMSE:", rmse(target_train, pred_train).round(5))
print("Valid RMSE:", rmse(target_valid, pred_valid).round(5))
print("Test RMSE: ", rmse(target_test, pred_test).round(5))


Train RMSE: 772.21538
Valid RMSE: 1729.6912
Test RMSE:  1728.29424


 **Gradient boosting LightGBM**

In [29]:
# %%

import lightgbm as lgb

model = lgb.LGBMRegressor(num_iterations=1000, vebose=1, metric='rmse')
model.fit(features_train, target_train, 
          eval_set=(features_valid, target_valid),
          categorical_feature=categorical_features)


/Users/juliokorleone/anaconda3/lib/python3.10/site-packages/lightgbm/engine.py:177: UserWarning: Found `num_iterations` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")
/Users/juliokorleone/anaconda3/lib/python3.10/site-packages/lightgbm/basic.py:2065: UserWarning: Using categorical_feature in Dataset.
  _log_warning('Using categorical_feature in Dataset.')
/Users/juliokorleone/anaconda3/lib/python3.10/site-packages/lightgbm/basic.py:2068: UserWarning: categorical_feature in Dataset is overridden.
New categorical_feature is ['Brand', 'FuelType', 'Gearbox', 'Model', 'Repaired', 'VehicleType']
  _log_warning('categorical_feature in Dataset is overridden.\n'
/Users/juliokorleone/anaconda3/lib/python3.10/site-packages/lightgbm/basic.py:1780: UserWarning: Overriding the parameters from Reference Dataset.
  _log_warning('Overriding the parameters from Reference Dataset.')
/Users/juliokorleone/anaconda3/lib/python3.10/sit

[LightGBM] [Warning] Unknown parameter: vebose
[1]	valid_0's rmse: 4264.82
[2]	valid_0's rmse: 3979.81
[3]	valid_0's rmse: 3731.85
[4]	valid_0's rmse: 3510.76
[5]	valid_0's rmse: 3314.92
[6]	valid_0's rmse: 3143.46
[7]	valid_0's rmse: 2990.73
[8]	valid_0's rmse: 2853.47
[9]	valid_0's rmse: 2736.16
[10]	valid_0's rmse: 2628.56
[11]	valid_0's rmse: 2536.41
[12]	valid_0's rmse: 2453.54
[13]	valid_0's rmse: 2380.55
[14]	valid_0's rmse: 2316.78
[15]	valid_0's rmse: 2259.17
[16]	valid_0's rmse: 2210.47
[17]	valid_0's rmse: 2167.23
[18]	valid_0's rmse: 2129.88
[19]	valid_0's rmse: 2093.37
[20]	valid_0's rmse: 2062.17
[21]	valid_0's rmse: 2034.07
[22]	valid_0's rmse: 2009.03
[23]	valid_0's rmse: 1987.14
[24]	valid_0's rmse: 1969.08
[25]	valid_0's rmse: 1952.24
[26]	valid_0's rmse: 1936.99
[27]	valid_0's rmse: 1923.5
[28]	valid_0's rmse: 1911.9
[29]	valid_0's rmse: 1901.18
[30]	valid_0's rmse: 1891.88
[31]	valid_0's rmse: 1883.81
[32]	valid_0's rmse: 1875.78
[33]	valid_0's rmse: 1868.09
[34]	va

LGBMRegressor(metric='rmse', num_iterations=1000, vebose=1)

In [30]:
# %%

pred_train = model.predict(features_train)
pred_valid = model.predict(features_valid)
pred_test = model.predict(features_test)


In [31]:
# %%
print("Train RMSE:", rmse(target_train, pred_train).round(5))
print("Valid RMSE:", rmse(target_valid, pred_valid).round(5))
print("Test RMSE: ", rmse(target_test, pred_test).round(5))


Train RMSE: 1410.16135
Valid RMSE: 1650.98246
Test RMSE:  1643.3866


In [32]:
# %%

import lightgbm as lgb

model = lgb.LGBMRegressor(num_iterations=1000, vebose=1, metric='rmse')
model.fit(features_train, target_train, 
          eval_set=(features_valid, target_valid))


[LightGBM] [Warning] Unknown parameter: vebose
[1]	valid_0's rmse: 4268.81
[2]	valid_0's rmse: 3988.72
[3]	valid_0's rmse: 3745.28
[4]	valid_0's rmse: 3533.96
[5]	valid_0's rmse: 3344.79
[6]	valid_0's rmse: 3180.15
[7]	valid_0's rmse: 3038.53
[8]	valid_0's rmse: 2913.76
[9]	valid_0's rmse: 2800.89
[10]	valid_0's rmse: 2703.65
[11]	valid_0's rmse: 2623.15
[12]	valid_0's rmse: 2548.09
[13]	valid_0's rmse: 2484.49
[14]	valid_0's rmse: 2428.51
[15]	valid_0's rmse: 2376.84
[16]	valid_0's rmse: 2330.6
[17]	valid_0's rmse: 2291.97
[18]	valid_0's rmse: 2256.88
[19]	valid_0's rmse: 2225.96
[20]	valid_0's rmse: 2197.94


/Users/juliokorleone/anaconda3/lib/python3.10/site-packages/lightgbm/engine.py:177: UserWarning: Found `num_iterations` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")


[21]	valid_0's rmse: 2172.21
[22]	valid_0's rmse: 2146.89
[23]	valid_0's rmse: 2125.14
[24]	valid_0's rmse: 2105.92
[25]	valid_0's rmse: 2085.68
[26]	valid_0's rmse: 2069.76
[27]	valid_0's rmse: 2055.27
[28]	valid_0's rmse: 2041.37
[29]	valid_0's rmse: 2030.06
[30]	valid_0's rmse: 2019.88
[31]	valid_0's rmse: 2009.93
[32]	valid_0's rmse: 1999.63
[33]	valid_0's rmse: 1990.62
[34]	valid_0's rmse: 1984.02
[35]	valid_0's rmse: 1974.87
[36]	valid_0's rmse: 1967.7
[37]	valid_0's rmse: 1959.75
[38]	valid_0's rmse: 1952.77
[39]	valid_0's rmse: 1945.53
[40]	valid_0's rmse: 1940.22
[41]	valid_0's rmse: 1935.15
[42]	valid_0's rmse: 1930.31
[43]	valid_0's rmse: 1925.69
[44]	valid_0's rmse: 1920.23
[45]	valid_0's rmse: 1914.48
[46]	valid_0's rmse: 1910.76
[47]	valid_0's rmse: 1907.91
[48]	valid_0's rmse: 1903.88
[49]	valid_0's rmse: 1901.23
[50]	valid_0's rmse: 1898.23
[51]	valid_0's rmse: 1895.32
[52]	valid_0's rmse: 1890.67
[53]	valid_0's rmse: 1886.49
[54]	valid_0's rmse: 1884.32
[55]	valid_0's 

LGBMRegressor(metric='rmse', num_iterations=1000, vebose=1)

In [33]:
# %%

pred_train = model.predict(features_train)
pred_valid = model.predict(features_valid)
pred_test = model.predict(features_test)


In [34]:
# %%
print("Train RMSE:", rmse(target_train, pred_train).round(5))
print("Valid RMSE:", rmse(target_valid, pred_valid).round(5))
print("Test RMSE: ", rmse(target_test, pred_test).round(5))


Train RMSE: 1460.69366
Valid RMSE: 1674.35245
Test RMSE:  1657.71951


 **Gradient boosting CatBoost**

In [40]:
# %%

from catboost import CatBoostRegressor

model = CatBoostRegressor(iterations=1000,
                          learning_rate=0.1,
                          #cat_features=data[categorical_features].dropna(), # TODO FIXME Invalid type for cat_feature[non-default value idx=0,feature_idx=0]=4.0 : cat_features must be integer or string, real number values and NaN values should be converted to string.
                          metric_period=50)
model.fit(features_train, target_train, 
          eval_set=(features_valid, target_valid))


0:	learn: 4303.6641569	test: 4301.7128896	best: 4301.7128896 (0)	total: 7.38ms	remaining: 7.37s
50:	learn: 2038.0178504	test: 2040.6427778	best: 2040.6427778 (50)	total: 310ms	remaining: 5.78s
100:	learn: 1916.1013000	test: 1924.7826614	best: 1924.7826614 (100)	total: 565ms	remaining: 5.03s
150:	learn: 1861.0469932	test: 1875.5131021	best: 1875.5131021 (150)	total: 824ms	remaining: 4.63s
200:	learn: 1825.3551799	test: 1846.3659885	best: 1846.3659885 (200)	total: 1.09s	remaining: 4.33s
250:	learn: 1798.6495174	test: 1824.8524135	best: 1824.8524135 (250)	total: 1.34s	remaining: 4s
300:	learn: 1778.0522977	test: 1809.5490852	best: 1809.5490852 (300)	total: 1.58s	remaining: 3.67s
350:	learn: 1759.2884167	test: 1795.8827617	best: 1795.8827617 (350)	total: 1.82s	remaining: 3.36s
400:	learn: 1743.8840102	test: 1785.3495521	best: 1785.3495521 (400)	total: 2.06s	remaining: 3.08s
450:	learn: 1730.0319854	test: 1775.2968687	best: 1775.2968687 (450)	total: 2.3s	remaining: 2.8s
500:	learn: 1717.615

In [41]:
# %%

pred_train = model.predict(features_train)
pred_valid = model.predict(features_valid)
pred_test = model.predict(features_test)


In [42]:
# %%
print("Train RMSE:", rmse(target_train, pred_train).round(5))
print("Valid RMSE:", rmse(target_valid, pred_valid).round(5))
print("Test RMSE: ", rmse(target_test, pred_test).round(5))


Train RMSE: 1636.87506
Valid RMSE: 1724.40131
Test RMSE:  1711.01931


In [43]:
# %%

from catboost import CatBoostRegressor

model = CatBoostRegressor(iterations=1000,
                          learning_rate=0.1,
                          metric_period=50)
model.fit(features_train, target_train, 
          eval_set=(features_valid, target_valid))


0:	learn: 4303.6641569	test: 4301.7128896	best: 4301.7128896 (0)	total: 6.1ms	remaining: 6.09s
50:	learn: 2038.0178504	test: 2040.6427778	best: 2040.6427778 (50)	total: 253ms	remaining: 4.7s
100:	learn: 1916.1013000	test: 1924.7826614	best: 1924.7826614 (100)	total: 496ms	remaining: 4.41s
150:	learn: 1861.0469932	test: 1875.5131021	best: 1875.5131021 (150)	total: 757ms	remaining: 4.25s
200:	learn: 1825.3551799	test: 1846.3659885	best: 1846.3659885 (200)	total: 1.01s	remaining: 4.03s
250:	learn: 1798.6495174	test: 1824.8524135	best: 1824.8524135 (250)	total: 1.28s	remaining: 3.81s
300:	learn: 1778.0522977	test: 1809.5490852	best: 1809.5490852 (300)	total: 1.52s	remaining: 3.54s
350:	learn: 1759.2884167	test: 1795.8827617	best: 1795.8827617 (350)	total: 1.79s	remaining: 3.31s
400:	learn: 1743.8840102	test: 1785.3495521	best: 1785.3495521 (400)	total: 2.04s	remaining: 3.05s
450:	learn: 1730.0319854	test: 1775.2968687	best: 1775.2968687 (450)	total: 2.29s	remaining: 2.79s
500:	learn: 1717.

In [44]:
# %%

pred_train = model.predict(features_train)
pred_valid = model.predict(features_valid)
pred_test = model.predict(features_test)


In [45]:
# %%
print("Train RMSE:", rmse(target_train, pred_train).round(5))
print("Valid RMSE:", rmse(target_valid, pred_valid).round(5))
print("Test RMSE: ", rmse(target_test, pred_test).round(5))


Train RMSE: 1636.87506
Valid RMSE: 1724.40131
Test RMSE:  1711.01931


 # 3. Model analysis

 | Model | RMSE | Prediction time, с | Training time, with |

 |----|----|-------|--------|

 | Линейная регрессия | 3220 | 0.156 | 0.4 |

 | LightGBM(1000) | 1640 | 8.47 | 12.9 |

 | CatBoost(1000) | 1730 | 0.899 | 27.2 |





 - The fastest (in training and prediction) model is linear regression. But the quality is low

 - The model with the best quality is gradient boosting LightGBM with 1000 trees.

 - The model with balanced prediction speed and quality is gradient boosting CatBoost with 1000 trees without category processing

In [46]:
# %%



